System information (for reproducibility):

In [46]:
versioninfo()

Julia Version 1.12.5
Commit 5fe89b8ddc1 (2026-02-09 16:05 UTC)
Build Info:
  Official https://julialang.org release
Platform Info:
  OS: macOS (arm64-apple-darwin24.0.0)
  CPU: 12 × Apple M2 Max
  WORD_SIZE: 64
  LLVM: libLLVM-18.1.7 (ORCJIT, apple-m2)
  GC: Built with stock GC
Threads: 8 default, 1 interactive, 8 GC (on 8 virtual cores)
Environment:
  JULIA_NUM_THREADS = 8
  JULIA_EDITOR = code


Load packages:

In [47]:
using Pkg

Pkg.activate(pwd())
Pkg.instantiate()
Pkg.status()

  Activating project at `~/Documents/github.com/ucla-biostat-257/2026spring/slides/08-numalgintro`


Status `~/Documents/github.com/ucla-biostat-257/2026spring/slides/08-numalgintro/Project.toml`
  [6e4b80f9] BenchmarkTools v1.7.0
  [0e44f5e4] Hwloc v3.3.0
  [bdcacae8] LoopVectorization v0.12.173
  [6f49c342] RCall v0.14.12
  [37e2e46d] LinearAlgebra v1.12.0
  [9a3f8284] Random v1.11.0


## Introduction

* Topics in numerical algebra: 
    - BLAS  
    - solve linear equations $\mathbf{A} \mathbf{x} = \mathbf{b}$
    - regression computations $\mathbf{X}^T \mathbf{X} \beta = \mathbf{X}^T \mathbf{y}$  
    - eigen-problems $\mathbf{A} \mathbf{x} = \lambda \mathbf{x}$  
    - generalized eigen-problems $\mathbf{A} \mathbf{x} = \lambda \mathbf{B} \mathbf{x}$  
    - singular value decompositions $\mathbf{A} = \mathbf{U} \Sigma \mathbf{V}^T$  
    - iterative methods for numerical linear algebra    

* Except for the iterative methods, most of these numerical linear algebra tasks are implemented in the **BLAS** and **LAPACK** libraries. They form the **building blocks** of most statistical computing tasks (optimization, MCMC).

* Our major **goal** (or learning objectives) is to  
    1. know the complexity (flop count) of each task
    2. be familiar with the BLAS and LAPACK functions (what they do)  
    3. do **not** re-invent wheels by implementing these dense linear algebra subroutines by yourself  
    4. understand the need for iterative methods  
    5. apply appropriate numerical algebra tools to various statistical problems 

* All high-level languages (Julia, Matlab, Python, R) call BLAS and LAPACK for numerical linear algebra. 
    - Julia offers more flexibility by exposing interfaces to many BLAS/LAPACK subroutines directly. See [documentation](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#BLAS-functions-1).

## BLAS

* BLAS stands for _basic linear algebra subprograms_. 

* See [netlib](http://www.netlib.org/blas/) for a complete list of standardized BLAS functions.

* There are many implementations of BLAS. 
    - [Netlib](http://www.netlib.org/blas/) provides a reference implementation.  
    - Matlab uses Intel's [MKL](https://www.intel.com/content/www/us/en/docs/oneapi/programming-guide/2023-0/intel-oneapi-math-kernel-library-onemkl.html) (mathematical kernel libaries). **MKL implementation is the gold standard on market.** It is not open source but the compiled library is free for Linux and MacOS. However, not surprisingly, it only works on Intel CPUs.      
    - Julia uses [OpenBLAS](https://github.com/xianyi/OpenBLAS). **OpenBLAS is the best cross-platform, open source implementation**. With the [MKL.jl](https://github.com/JuliaLinearAlgebra/MKL.jl) package, it's also very easy to use MKL in Julia.    

* There are 3 levels of BLAS functions.
    - [Level 1](http://www.netlib.org/blas/#_level_1): vector-vector operation
    - [Level 2](http://www.netlib.org/blas/#_level_2): matrix-vector operation
    - [Level 3](http://www.netlib.org/blas/#_level_3): matrix-matrix operation

| Level | Example Operation                      | Name        | Dimension                                 | Flops |  
|-------|----------------------------------------|-------------|-------------------------------------------|-------|
| 1     | $\alpha \gets \mathbf{x}^T \mathbf{y}$ | dot product | $\mathbf{x}, \mathbf{y} \in \mathbb{R}^n$ | $2n$  |  
| 1 | $\mathbf{y} \gets \mathbf{y} + \alpha \mathbf{x}$ |  axpy           |  $\alpha \in \mathbb{R}$, $\mathbf{x}, \mathbf{y} \in \mathbb{R}^n$ |  $2n$    |  
| 2     | $\mathbf{y} \gets \mathbf{y} + \mathbf{A} \mathbf{x}$ |  gaxpy           |  $\mathbf{A} \in \mathbb{R}^{m \times n}$, $\mathbf{x} \in \mathbb{R}^n$, $\mathbf{y} \in \mathbb{R}^m$                                     |  $2mn$     |
| 2 | $\mathbf{A} \gets \mathbf{A} + \mathbf{y} \mathbf{x}^T$ | rank one update            |    $\mathbf{A} \in \mathbb{R}^{m \times n}$, $\mathbf{x} \in \mathbb{R}^n$, $\mathbf{y} \in \mathbb{R}^m$                                       | $2mn$      |
| 3     | $\mathbf{C} \gets \mathbf{C} + \mathbf{A} \mathbf{B}$                                       |  matrix multiplication           |  $\mathbf{A} \in \mathbb{R}^{m \times p}$, $\mathbf{B} \in \mathbb{R}^{p \times n}$, $\mathbf{C} \in \mathbb{R}^{m \times n}$                                         | $2mnp$      |

* Typical BLAS functions support single precision (S), double precision (D), complex (C), and double complex (Z). 

## Examples

> **The form of a mathematical expression and the way the expression should be evaluated in actual practice may be quite different.**

Some operations _appear_ as level-3 but indeed are level-2.  

**Example 1**. A common operation in statistics is column scaling or row scaling
$$
\begin{eqnarray*}
    \mathbf{A} &=& \mathbf{A} \mathbf{D} \quad \text{(column scaling)} \\
    \mathbf{A} &=& \mathbf{D} \mathbf{A} \quad \text{(row scaling)},
\end{eqnarray*}
$$
where $\mathbf{D}$ is diagonal. For example, in generalized linear models (GLMs), the Fisher information matrix takes the form  
$$
\mathbf{X}^T \mathbf{W} \mathbf{X},
$$
where $\mathbf{W}$ is a diagonal matrix with observation weights on diagonal.  

  Column and row scalings are essentially level-2 operations!

In [48]:
using BenchmarkTools, LinearAlgebra, Random

Random.seed!(257) # seed
n = 2000
A = rand(n, n) # n-by-n matrix
d = rand(n)  # n vector
D = Diagonal(d) # diagonal matrix with d as diagonal

2000×2000 Diagonal{Float64, Vector{Float64}}:
 0.0416032   ⋅         ⋅         ⋅       …   ⋅         ⋅         ⋅ 
  ⋅         0.887679   ⋅         ⋅           ⋅         ⋅         ⋅ 
  ⋅          ⋅        0.102233   ⋅           ⋅         ⋅         ⋅ 
  ⋅          ⋅         ⋅        0.08407      ⋅         ⋅         ⋅ 
  ⋅          ⋅         ⋅         ⋅           ⋅         ⋅         ⋅ 
  ⋅          ⋅         ⋅         ⋅       …   ⋅         ⋅         ⋅ 
  ⋅          ⋅         ⋅         ⋅           ⋅         ⋅         ⋅ 
  ⋅          ⋅         ⋅         ⋅           ⋅         ⋅         ⋅ 
  ⋅          ⋅         ⋅         ⋅           ⋅         ⋅         ⋅ 
  ⋅          ⋅         ⋅         ⋅           ⋅         ⋅         ⋅ 
 ⋮                                       ⋱                      
  ⋅          ⋅         ⋅         ⋅           ⋅         ⋅         ⋅ 
  ⋅          ⋅         ⋅         ⋅           ⋅         ⋅         ⋅ 
  ⋅          ⋅         ⋅         ⋅           ⋅         ⋅         ⋅ 
  ⋅  

In [49]:
Dfull = diagm(d) # convert to full matrix

2000×2000 Matrix{Float64}:
 0.0416032  0.0       0.0       0.0      …  0.0       0.0       0.0
 0.0        0.887679  0.0       0.0         0.0       0.0       0.0
 0.0        0.0       0.102233  0.0         0.0       0.0       0.0
 0.0        0.0       0.0       0.08407     0.0       0.0       0.0
 0.0        0.0       0.0       0.0         0.0       0.0       0.0
 0.0        0.0       0.0       0.0      …  0.0       0.0       0.0
 0.0        0.0       0.0       0.0         0.0       0.0       0.0
 0.0        0.0       0.0       0.0         0.0       0.0       0.0
 0.0        0.0       0.0       0.0         0.0       0.0       0.0
 0.0        0.0       0.0       0.0         0.0       0.0       0.0
 ⋮                                       ⋱                      
 0.0        0.0       0.0       0.0         0.0       0.0       0.0
 0.0        0.0       0.0       0.0         0.0       0.0       0.0
 0.0        0.0       0.0       0.0         0.0       0.0       0.0
 0.0        0.0       0.

In [50]:
# this is calling BLAS routine for matrix multiplication: O(n^3) flops
# this is SLOW!
@benchmark $A * $Dfull

BenchmarkTools.Trial: 105 samples with 1 evaluation per sample.
 Range (min … max):  44.892 ms … 57.043 ms  ┊ GC (min … max): 0.00% … 1.23%
 Time  (median):     46.715 ms              ┊ GC (median):    1.53%
 Time  (mean ± σ):   47.889 ms ±  2.744 ms  ┊ GC (mean ± σ):  1.28% ± 0.75%

     ▂▃▁█                                                      
  ▄▃▃████▇▃▇▅▄▅▄▄▃▃▇▅▄▃▁▃▃▁▃▃▁▃▅▁▃▃▃▄▃▁▃▃▃▃▁▃▃▁▃▁▃▁▁▃▁▁▁▁▁▄▁▃ ▃
  44.9 ms         Histogram: frequency by time        55.7 ms <

 Memory estimate: 30.53 MiB, allocs estimate: 3.

In [51]:
# dispatch to special method for diagonal matrix multiplication.
# columnwise scaling: O(n^2) flops
@benchmark $A * $D

BenchmarkTools.Trial: 4022 samples with 1 evaluation per sample.
 Range (min … max):  735.708 μs …   2.168 ms  ┊ GC (min … max):  0.00% … 32.60%
 Time  (median):       1.065 ms               ┊ GC (median):     0.00%
 Time  (mean ± σ):     1.240 ms ± 309.186 μs  ┊ GC (mean ± σ):  16.45% ± 17.44%

            ▇█▅▆▆▆▅▄▃▂▁▁  ▁ ▁▂▁▁       ▂▄▅▅▅▄▄▄▃▃▃▂▁            ▂
  ▄▄▁▁▁▃▁▃▆██████████████████████▅▄▄▁▄▇██████████████▇██▆▇▇█▇▆█ █
  736 μs        Histogram: log(frequency) by time          2 ms <

 Memory estimate: 30.53 MiB, allocs estimate: 3.

In [52]:
# Or we can use broadcasting (with recycling)
@benchmark $A .* transpose($d)

BenchmarkTools.Trial: 4010 samples with 1 evaluation per sample.
 Range (min … max):  748.834 μs …   2.252 ms  ┊ GC (min … max):  0.00% … 36.61%
 Time  (median):       1.063 ms               ┊ GC (median):     0.00%
 Time  (mean ± σ):     1.243 ms ± 314.262 μs  ┊ GC (mean ± σ):  16.57% ± 17.59%

           ▆█▆▅▅▅▅▄▃▂  ▁  ▁▂▂▁▁       ▂▄▅▅▄▄▄▃▃▃▂ ▁             ▂
  ▄▁▁▃▃▃▅▆██████████████████████▇▄▄▅▅▆███████████▇██▇█▇█▇█▇▇▇██ █
  749 μs        Histogram: log(frequency) by time       2.03 ms <

 Memory estimate: 30.53 MiB, allocs estimate: 3.

In [53]:
# in-place: avoid allocate space for result
# rmul!: compute matrix-matrix product A*B, overwriting A, and return the result.
@benchmark rmul!($A, $D)

BenchmarkTools.Trial: 8561 samples with 1 evaluation per sample.
 Range (min … max):  530.792 μs …   8.126 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     556.750 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   579.507 μs ± 194.380 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

      █▇▇▆▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁                                   ▂
  ▄▁▁▄█████████████████████████▆▇▆▆▆▆▅▄▆▆▅▅▆▄▅▆▅▆▅▅▅▆▆▆▇▆▆▇▇▇▆▇ █
  531 μs        Histogram: log(frequency) by time        779 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [54]:
# In-place broadcasting 
@benchmark $A .= $A .* transpose($d)

BenchmarkTools.Trial: 3009 samples with 1 evaluation per sample.
 Range (min … max):  1.484 ms …   9.724 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     1.567 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   1.659 ms ± 516.878 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

   █▆▅▄▃                                                      ▁
  ███████▇█▇▇▃▄▃▅▃▁▁▁▄▁▅▅▃▁▄▅▁▅▁▃▃▃▁▃▁▁▁▁▄▁▃▁▄▁▄▃▃▁▃▃▁▃▄▁▁▃▄▄ █
  1.48 ms      Histogram: log(frequency) by time      4.09 ms <

 Memory estimate: 0 bytes, allocs estimate: 0.

**Exercise**: Try `@turbo` (SIMD) and `@tturbo` (multi-threaded SIMD) from LoopVectorization.jl package.

**Note:** In R or Matlab, `diag(d)` will create a full matrix. Be cautious using `diag` function: do we really need a full diagonal matrix?

In [55]:
using RCall

R"""
d <- runif(5)
diag(d)
"""

RObject{RealSxp}
          [,1]      [,2]      [,3]      [,4]      [,5]
[1,] 0.3047994 0.0000000 0.0000000 0.0000000 0.0000000
[2,] 0.0000000 0.1219085 0.0000000 0.0000000 0.0000000
[3,] 0.0000000 0.0000000 0.7471399 0.0000000 0.0000000
[4,] 0.0000000 0.0000000 0.0000000 0.6733713 0.0000000
[5,] 0.0000000 0.0000000 0.0000000 0.0000000 0.8882592


This works only when Matlab is installed.

```julia
#| eval: false
using MATLAB

mat"""
d = rand(5, 1)
diag(d)
```

**Example 2**. Innter product between two matrices $\mathbf{A}, \mathbf{B} \in \mathbb{R}^{m \times n}$ is often written as 
$$
    \text{trace}(\mathbf{A}^T \mathbf{B}), \text{trace}(\mathbf{B} \mathbf{A}^T), \text{trace}(\mathbf{A} \mathbf{B}^T), \text{ or } \text{trace}(\mathbf{B}^T \mathbf{A}).
$$
They appear as level-3 operation (matrix multiplication with $O(m^2n)$ or $O(mn^2)$ flops).

In [56]:
Random.seed!(123)
n = 2000
A, B = randn(n, n), randn(n, n)

# slow way to evaluate tr(A'B): 2mn^2 flops
@benchmark tr(transpose($A) * $B)

BenchmarkTools.Trial: 105 samples with 1 evaluation per sample.
 Range (min … max):  44.461 ms … 56.994 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     46.707 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   47.607 ms ±  2.878 ms  ┊ GC (mean ± σ):  0.70% ± 0.85%

    █▄▃▄█▁▃ ▁ ▃ ▃    ▁    ▁                                    
  ▆▇███████▄█▄█▆█▆▇▆▄█▄▁▁▁█▄▁▄▄▁▁▄▄▄▄▄▄▄▁▄▄▁▄▁▄▆▆▄▁▁▄▄▁▁▁▁▁▁▄ ▄
  44.5 ms         Histogram: frequency by time        55.7 ms <

 Memory estimate: 30.53 MiB, allocs estimate: 3.

But $\text{trace}(\mathbf{A}^T \mathbf{B}) = <\text{vec}(\mathbf{A}), \text{vec}(\mathbf{B})>$. The latter is level-1 BLAS operation with $O(mn)$ flops.

In [57]:
# smarter way to evaluate tr(A'B): 2mn flops
@benchmark dot($A, $B)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  403.125 μs …   5.493 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     448.958 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   462.422 μs ± 101.955 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

     ▄█▅▄▂▃▁    ▂▂▁                                              
  ▂▂▄███████████████▇▆▅▄▄▄▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂ ▄
  403 μs           Histogram: frequency by time          647 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

**Example 3**. Similarly $\text{diag}(\mathbf{A}^T \mathbf{B})$ can be calculated in $O(mn)$ flops.

In [58]:
# slow way to evaluate diag(A'B): O(n^3)
@benchmark diag(transpose($A) * $B)

BenchmarkTools.Trial: 100 samples with 1 evaluation per sample.
 Range (min … max):  46.665 ms … 58.377 ms  ┊ GC (min … max): 0.00% … 1.28%
 Time  (median):     49.284 ms              ┊ GC (median):    1.57%
 Time  (mean ± σ):   50.112 ms ±  2.638 ms  ┊ GC (mean ± σ):  1.30% ± 0.78%

         ▆█▆▆  ▁▄         ▄             ▃                      
  ▆▄▆▆▆▇▄████▆▆██▇▄▆▆▇▁▆▁▆█▄▆▁▁▄▄▆▄▄▁▁▁▁█▄▁▁▁▁▁▆▁▁▁▁▄▁▁▆▁▁▄▁▄ ▄
  46.7 ms         Histogram: frequency by time        57.7 ms <

 Memory estimate: 30.55 MiB, allocs estimate: 6.

In [59]:
# smarter way to evaluate diag(A'B): O(n^2)
@benchmark Diagonal(vec(sum($A .* $B, dims = 1)))

BenchmarkTools.Trial: 2407 samples with 1 evaluation per sample.
 Range (min … max):  1.569 ms …  37.295 ms  ┊ GC (min … max):  0.00% … 95.31%
 Time  (median):     1.875 ms               ┊ GC (median):     0.00%
 Time  (mean ± σ):   2.069 ms ± 840.975 μs  ┊ GC (mean ± σ):  13.37% ± 13.54%

   ▂█▂▁▁▂▃▃▁                                                   
  ▂██████████▅▅▃▃▂▂▂▂▂▂▂▂▂▂▂▃▅▇▆▇▆▇▇▇▆▅▄▄▃▃▃▃▂▃▂▂▂▁▂▂▂▂▂▂▂▂▁▁ ▃
  1.57 ms         Histogram: frequency by time        3.02 ms <

 Memory estimate: 30.55 MiB, allocs estimate: 7.

To get rid of allocation of intermediate arrays at all, we can just write a double loop or use `dot` function.

In [60]:
function diag_matmul!(d, A, B)
    m, n = size(A)
    @assert size(B) == (m, n) "A and B should have same size"
    fill!(d, 0)
    for j in 1:n, i in 1:m
        d[j] += A[i, j] * B[i, j]
    end
    Diagonal(d)
end

d = zeros(eltype(A), size(A, 2))
@benchmark diag_matmul!($d, $A, $B)

BenchmarkTools.Trial: 1450 samples with 1 evaluation per sample.
 Range (min … max):  3.341 ms …   4.100 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     3.410 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   3.445 ms ± 119.721 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▃                                                           
  ██▇▆▇▇▇▆▅▅▅▅▆▅▅▅▄▅▄▃▄▃▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▂▂▂▂▂▂▁▂▂▂▂▂▂▂▂▂▂▂▂▂ ▃
  3.34 ms         Histogram: frequency by time        3.95 ms <

 Memory estimate: 0 bytes, allocs estimate: 0.

**Exercise**: Try `@turbo` (SIMD) and `@tturbo` (multi-threaded SIMD) from LoopVectorization.jl package.

## Memory hierarchy and level-3 fraction

> **Key to high performance is effective use of memory hierarchy. True on all architectures.**

* Flop count is not the sole determinant of algorithm efficiency. Another important factor is data movement through the memory hierarchy.

<img src="./cpu_die.png" width="400" align="center">

<img src="./macpro_inside.png" width="400" align="center">  

<img src="./cache_speed.png" width="600" align="center">

Source: <https://cs.brown.edu/courses/csci1310/2020/assign/labs/lab4.html>

- In Julia, we can query the CPU topology by the `Hwloc.jl` package. For example, this laptop runs an Apple M2 Max chip with 4 efficiency cores and 8 performance cores.

In [61]:
using Hwloc

topology_graphical()

┌──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ Machine (96GB total)                                                                                                                                             │
│                                                                                                                                                                  │
│ ┌────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐  ┌──────────────────┐ │
│ │ Package L#0                                                                                                                            │  │ CoProc opencl0d0 │ │
│ │                                                                                                                                        │  │                  │ │
│ │ ┌─────

* For example, Xeon X5650 CPU has a theoretical throughput of 128 DP GFLOPS but a max memory bandwidth of 32GB/s.  

* Can we keep CPU cores busy with enough deliveries of matrix data and ship the results to memory fast enough to avoid backlog?  
Answer: use **high-level BLAS** as much as possible.

| BLAS | Dimension | Mem. Refs. | Flops  | Ratio |
|--------------------------------|------------------------------------------------------------|------------|--------|-------|
| Level 1: $\mathbf{y} \gets \mathbf{y} + \alpha \mathbf{x}$     | $\mathbf{x}, \mathbf{y} \in \mathbb{R}^n$                                           | $3n$       | $2n$   | 3:2   |
| Level 2: $\mathbf{y} \gets \mathbf{y} + \mathbf{A} \mathbf{x}$ | $\mathbf{x}, \mathbf{y} \in \mathbb{R}^n$, $\mathbf{A} \in \mathbb{R}^{n \times n}$ | $n^2$      | $2n^2$ | 1:2   |
| Level 3: $\mathbf{C} \gets \mathbf{C} + \mathbf{A} \mathbf{B}$ | $\mathbf{A}, \mathbf{B}, \mathbf{C} \in\mathbb{R}^{n \times n}$                    | $4n^2$     | $2n^3$ | 2:n |  

* Higher level BLAS (3 or 2) make more effective use of arithmetic logic units (ALU) by keeping them busy. **Surface-to-volume** effect.  

<img src="./blas_throughput.png" width="500" align="center"/>

Source: [Jack Dongarra's slides](https://raw.githubusercontent.com/ucla-biostat-257/2023spring/master/readings/SAMSI-0217_Dongarra.pdf).

* A distinction between LAPACK and LINPACK (older version of R uses LINPACK) is that LAPACK makes use of higher level BLAS as much as possible (usually by smart partitioning) to increase the so-called **level-3 fraction**.

* To appreciate the efforts in an optimized BLAS implementation such as OpenBLAS (evolved from GotoBLAS), see the [Quora question](https://www.quora.com/What-algorithm-does-BLAS-use-for-matrix-multiplication-Of-all-the-considerations-e-g-cache-popular-instruction-sets-Big-O-etc-which-one-turned-out-to-be-the-primary-bottleneck), especially the [video](https://youtu.be/JzNpKDW07rw). Bottomline is 

> **Get familiar with (good implementations of) BLAS/LAPACK and use them as much as possible.**

## Effect of data layout

* Data layout in memory affects algorithmic efficiency too. It is much faster to move chunks of data in memory than retrieving/writing scattered data.

* Storage mode: **column-major** (Fortran, Matlab, R, Julia) vs **row-major** (Python numpy, C/C++).

* **Cache line** is the minimum amount of cache which can be loaded and stored to memory.
    - x86 CPUs: 64 bytes  
    - ARM CPUs: 32 bytes

<img src="https://patterns.eecs.berkeley.edu/wordpress/wp-content/uploads/2013/04/dense02.png" width="500" align="center"/>

* In Julia, we can query the cache line size by Hwloc.jl.

In [62]:
# Apple Silicon (M1/M2 chips) don't have L3 cache
Hwloc.cachelinesize()

ErrorException: Your system doesn't seem to have an L3 cache.

* Accessing column-major stored matrix by rows ($ij$ looping) causes lots of **cache misses**.

* Take matrix multiplication as an example 
$$ 
\mathbf{C} \gets \mathbf{C} + \mathbf{A} \mathbf{B}, \quad \mathbf{A} \in \mathbb{R}^{m \times p}, \mathbf{B} \in \mathbb{R}^{p \times n}, \mathbf{C} \in \mathbb{R}^{m \times n}.
$$
Assume the storage is column-major, such as in Julia. There are 6 variants of the algorithms according to the order in the triple loops. 

    - `jki` or `kji` looping:
    
```julia
# inner most loop
for i in 1:m
    C[i, j] = C[i, j] + A[i, k] * B[k, j]
end
```
        
    - `ikj` or `kij` looping:

```julia
# inner most loop        
for j in 1:n
    C[i, j] = C[i, j] + A[i, k] * B[k, j]
end
```  

- `ijk` or `jik` looping:

```julia
# inner most loop        
for k in 1:p
    C[i, j] = C[i, j] + A[i, k] * B[k, j]
end
```
        
* We pay attention to the innermost loop, where the vector calculation occurs. The associated **stride** when accessing the three matrices in memory (assuming column-major storage) is  

| Variant        | A Stride | B Stride | C Stride |
|----------------|----------|----------|----------|
| $jki$ or $kji$ | Unit     | 0        | Unit     |
| $ikj$ or $kij$ | 0        | Non-Unit | Non-Unit |
| $ijk$ or $jik$ | Non-Unit | Unit     | 0        |       
Apparently the variants $jki$ or $kji$ are preferred.

In [63]:
"""
    matmul_by_loop!(A, B, C, order)

Overwrite `C` by `A * B`. `order` indicates the looping order for triple loop.
"""
function matmul_by_loop!(A::Matrix, B::Matrix, C::Matrix, order::String)
    
    m = size(A, 1)
    p = size(A, 2)
    n = size(B, 2)
    fill!(C, 0)
    
    if order == "jki"
        @inbounds for j = 1:n, k = 1:p, i = 1:m
            C[i, j] += A[i, k] * B[k, j]
        end
    end

    if order == "kji"
        @inbounds for k = 1:p, j = 1:n, i = 1:m
            C[i, j] += A[i, k] * B[k, j]
        end
    end
    
    if order == "ikj"
        @inbounds for i = 1:m, k = 1:p, j = 1:n
            C[i, j] += A[i, k] * B[k, j]
        end
    end

    if order == "kij"
        @inbounds for k = 1:p, i = 1:m, j = 1:n
            C[i, j] += A[i, k] * B[k, j]
        end
    end
    
    if order == "ijk"
        @inbounds for i = 1:m, j = 1:n, k = 1:p
            C[i, j] += A[i, k] * B[k, j]
        end
    end
    
    if order == "jik"
        @inbounds for j = 1:n, i = 1:m, k = 1:p
            C[i, j] += A[i, k] * B[k, j]
        end
    end
    
end

using Random

Random.seed!(123)
m, p, n = 2000, 100, 2000
A = rand(m, p)
B = rand(p, n)
C = zeros(m, n);

* $jki$ and $kji$ looping:

In [64]:
using BenchmarkTools

@benchmark matmul_by_loop!($A, $B, $C, "jki")

BenchmarkTools.Trial: 86 samples with 1 evaluation per sample.
 Range (min … max):  57.762 ms …  60.251 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     58.277 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   58.541 ms ± 682.433 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

    ▁▁▆ █▃ ▃▃▁▃▁▆                                               
  ▇▇███▇██▇██████▄▁▇▇▄▇▄▇▁▄▁▁▇▁▁▁▁▁▁▁▇▇▄▄▁▇▇▄▁▁▁▁▁▁▄▁▄▇▁▇▁▁▁▄▄ ▁
  57.8 ms         Histogram: frequency by time         60.2 ms <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [65]:
@benchmark matmul_by_loop!($A, $B, $C, "kji")

BenchmarkTools.Trial: 59 samples with 1 evaluation per sample.
 Range (min … max):  81.778 ms … 127.982 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     82.665 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   84.852 ms ±   7.355 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▃█                                                            
  ███▄▄▇▃▄▄▁▃▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▁▁▁▁▁▁▁▁▁▁▁▃▁▁▁▁▁▁▁▁▁▁▃ ▁
  81.8 ms         Histogram: frequency by time          108 ms <

 Memory estimate: 0 bytes, allocs estimate: 0.

* $ikj$ and $kij$ looping:

In [66]:
@benchmark matmul_by_loop!($A, $B, $C, "ikj")

BenchmarkTools.Trial: 10 samples with 1 evaluation per sample.
 Range (min … max):  517.284 ms … 523.053 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     519.788 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   519.886 ms ±   1.823 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █     █         ██ █            ███                 █       █  
  █▁▁▁▁▁█▁▁▁▁▁▁▁▁▁██▁█▁▁▁▁▁▁▁▁▁▁▁▁███▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁█ ▁
  517 ms           Histogram: frequency by time          523 ms <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [67]:
@benchmark matmul_by_loop!($A, $B, $C, "kij")

BenchmarkTools.Trial: 10 samples with 1 evaluation per sample.
 Range (min … max):  522.803 ms … 532.528 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     527.831 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   528.114 ms ±   3.338 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █            █ █      █      █   █          █          █ █  █  
  █▁▁▁▁▁▁▁▁▁▁▁▁█▁█▁▁▁▁▁▁█▁▁▁▁▁▁█▁▁▁█▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁█▁█▁▁█ ▁
  523 ms           Histogram: frequency by time          533 ms <

 Memory estimate: 0 bytes, allocs estimate: 0.

* $ijk$ and $jik$ looping:

In [68]:
@benchmark matmul_by_loop!($A, $B, $C, "ijk")

BenchmarkTools.Trial: 21 samples with 1 evaluation per sample.
 Range (min … max):  241.644 ms … 252.364 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     245.913 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   245.943 ms ±   3.014 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▁  ▁▁█  ▁   █       ▁  ▁█ ▁       ▁▁   ▁█   ▁▁              ▁  
  █▁▁███▁▁█▁▁▁█▁▁▁▁▁▁▁█▁▁██▁█▁▁▁▁▁▁▁██▁▁▁██▁▁▁██▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  242 ms           Histogram: frequency by time          252 ms <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [69]:
@benchmark matmul_by_loop!($A, $B, $C, "ijk")

BenchmarkTools.Trial: 21 samples with 1 evaluation per sample.
 Range (min … max):  242.506 ms … 251.898 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     246.058 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   246.348 ms ±   2.846 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

           █▃                                                    
  ▇▇▁▁▁▁▇▁▇██▁▁▁▇▁▁▁▁▁▁▁▁▇▁▁▁▇▁▇▁▁▁▇▁▁▁▁▇▁▇▁▁▇▁▁▇▁▁▇▇▁▁▁▁▁▁▁▁▁▇ ▁
  243 ms           Histogram: frequency by time          252 ms <

 Memory estimate: 0 bytes, allocs estimate: 0.

* **Question: Can our loop beat BLAS?** Julia wraps BLAS library for matrix multiplication. We see BLAS library wins hands down (multi-threading, Strassen algorithm, higher level-3 fraction by block outer product).

In [70]:
@benchmark mul!($C, $A, $B)

BenchmarkTools.Trial: 1774 samples with 1 evaluation per sample.
 Range (min … max):  2.437 ms …   7.010 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     2.646 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   2.815 ms ± 495.116 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

    ▂█▇▄▃▄▃▂                                                   
  ▇▅█████████▇▇▄▆▄▇▆▆▆▆▇▇▆▇▆▆▅▅▇▆▅▅▆▅▅▆▅▃▅▄▃▃▅▅▃▄▁▁▃▃▁▄▄▁▁▃▁▃ █
  2.44 ms      Histogram: log(frequency) by time      5.38 ms <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [71]:
# direct call of BLAS wrapper function
@benchmark LinearAlgebra.BLAS.gemm!('N', 'N', 1.0, $A, $B, 0.0, $C)

BenchmarkTools.Trial: 1770 samples with 1 evaluation per sample.
 Range (min … max):  2.446 ms …   6.411 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     2.666 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   2.823 ms ± 466.498 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

    ▁▇█▇▅▄▄▃▂                                                 ▁
  ▅▄██████████▇██▇▇▆▆▆▆▆▄▇▇▇▅▆▆▇▆▆▆▆▆▆▄▄▆▅▆▄▅▄▄▅▄▄▄▅▄▆▄▁▅▄▄▄▄ █
  2.45 ms      Histogram: log(frequency) by time      5.04 ms <

 Memory estimate: 0 bytes, allocs estimate: 0.

**Question (again): Can our loop beat BLAS?**

**Exercise:** Annotate the loop in `matmul_by_loop!` by `@turbo` and `@tturbo` (multi-threading) and benchmark again.

## BLAS in R

* **Tip for R users**. Standard R distribution from CRAN uses a very out-dated BLAS/LAPACK library.

In [72]:
using RCall

R"""
sessionInfo()
"""

RObject{VecSxp}
R version 4.5.2 (2025-10-31)
Platform: aarch64-apple-darwin20
Running under: macOS Tahoe 26.4

Matrix products: default
BLAS:   /System/Library/Frameworks/Accelerate.framework/Versions/A/Frameworks/vecLib.framework/Versions/A/libBLAS.dylib 
LAPACK: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRlapack.dylib;  LAPACK version 3.12.1

locale:
[1] C.UTF-8/C.UTF-8/C.UTF-8/C/C.UTF-8/C.UTF-8

time zone: America/Los_Angeles
tzcode source: internal

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] bench_1.1.4 dplyr_1.1.4

loaded via a namespace (and not attached):
 [1] tidyselect_1.2.1 compiler_4.5.2   magrittr_2.0.4   R6_2.6.1        
 [5] generics_0.1.4   cli_3.6.5        pillar_1.11.1    glue_1.8.0      
 [9] tibble_3.3.1     utf8_1.2.6       vctrs_0.6.5      lifecycle_1.0.5 
[13] pkgconfig_2.0.3  rlang_1.1.7      profmem_0.7.0   


In [73]:
R"""
library(dplyr)
library(bench)
A <- $A
B <- $B
bench::mark(A %*% B) %>%
  print(width = Inf)
""";

# A tibble: 1 × 13
  expression      min   median `itr/sec` mem_alloc `gc/sec` n_itr  n_gc
  <bch:expr> <bch:tm> <bch:tm>     <dbl> <bch:byt>    <dbl> <int> <dbl>
1 A %*% B      1.87ms   1.91ms      481.    30.5MB     880.    64   117
  total_time result                memory             time            
    <bch:tm> <list>                <list>             <list>          
1      133ms <dbl [2,000 × 2,000]> <Rprofmem [1 × 3]> <bench_tm [181]>
  gc                
  <list>            
1 <tibble [181 × 3]>


* Re-build R from source using OpenBLAS or MKL will immediately boost linear algebra performance in R. Google `build R using MKL` to get started. Similarly we can build Julia using MKL.

* Matlab uses MKL. Usually it's very hard to beat Matlab in terms of linear algebra.

In [74]:
#| eval: false

using MATLAB

mat"""
f = @() $A * $B;
timeit(f)
"""

ArgumentError: ArgumentError: Package MATLAB not found in current path.
- Run `import Pkg; Pkg.add("MATLAB")` to install the MATLAB package.

## Avoid memory allocation: some examples

### Transposing matrix is an expensive memory operation

In R, the command 
```R
t(A) %*% x
```
will first transpose `A` then perform matrix multiplication, causing unnecessary memory allocation

In [75]:
using Random, LinearAlgebra, BenchmarkTools
Random.seed!(123)

n = 1000
A = rand(n, n)
x = rand(n);

In [76]:
R"""
A <- $A
x <- $x
bench::mark(t(A) %*% x) %>%
  print(width = Inf)
""";

# A tibble: 1 × 13
  expression      min   median `itr/sec` mem_alloc `gc/sec` n_itr  n_gc
  <bch:expr> <bch:tm> <bch:tm>     <dbl> <bch:byt>    <dbl> <int> <dbl>
1 t(A) %*% x   1.53ms   1.68ms      584.    7.64MB     71.8   244    30
  total_time result            memory             time            
    <bch:tm> <list>            <list>             <list>          
1      418ms <dbl [1,000 × 1]> <Rprofmem [2 × 3]> <bench_tm [274]>
  gc                
  <list>            
1 <tibble [274 × 3]>


Julia is avoids transposing matrix whenever possible. The `Transpose` type only creates a view of the transpose of matrix data.

In [77]:
typeof(transpose(A))

Transpose{Float64, Matrix{Float64}}

In [78]:
fieldnames(typeof(transpose(A)))

(:parent,)

In [79]:
# same data in tranpose(A) and original matrix A
pointer(transpose(A).parent), pointer(A)

(Ptr{Float64}(0x0000000b8ac00000), Ptr{Float64}(0x0000000b8ac00000))

In [80]:
# dispatch to BLAS
# does *not* actually transpose the matrix
@benchmark transpose($A) * $x

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  22.583 μs … 121.250 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     23.042 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   24.408 μs ±   6.469 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▆▁   ▁▂▂                                                    ▁
  ████████████▇▇██▇▆▅▅▅▄▆▅▅▄▃▃▄▄▁▃▁▃▁▁▄▃▃▄▄▁▅▅▃▅▄▄▅▅▇▇▇▆▆▅▅▅▅▆ █
  22.6 μs       Histogram: log(frequency) by time      55.4 μs <

 Memory estimate: 8.06 KiB, allocs estimate: 3.

In [81]:
# pre-allocate result
out = zeros(size(A, 2))
@benchmark mul!($out, transpose($A), $x)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  22.542 μs … 401.834 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     23.000 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   26.460 μs ±  11.961 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▁ ▁▂                      ▁▁                                ▁
  ██▇███▇██▆▄▅▅▄▄▂▄▄▄▃▃▄▄▂▆▇████▇▆▅▆▅▅▅▄▅▄▅▅▅▃▄▄▄▄▅▅▄▄▂▄▃▄▃▃▄▅ █
  22.5 μs       Histogram: log(frequency) by time      80.8 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [82]:
# or call BLAS wrapper directly
@benchmark LinearAlgebra.BLAS.gemv!('T', 1.0, $A, $x, 0.0, $out)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  22.625 μs … 436.667 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     22.917 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   25.226 μs ±  10.237 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▄▁    ▁                                     ▁▁▁             ▁
  ████▆▇███▇▆▇▇▇▆▅▆▄▃▆▄▄▄▄▁▄▄▃▄▄▄▄▄▄▅▄▃▄▄▃▄▅▅▅██████▇▆▆▅▆▅▅▅▄▅ █
  22.6 μs       Histogram: log(frequency) by time      58.5 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

### Broadcast (dot operation) fuses loops and avoids memory allocation

[Broadcasting](https://docs.julialang.org/en/v1/base/arrays/#Broadcast-and-vectorization-1) in Julia achieves vectorized code without creating intermediate arrays.

Suppose we want to calculate elementsize maximum of absolute values of two large arrays. In R or Matlab, the command
```r
max(abs(X), abs(Y))
```
will create two intermediate arrays and then one result array.

In [83]:
using RCall

Random.seed!(123)
X, Y = rand(1000, 1000), rand(1000, 1000)

R"""
library(dplyr)
library(bench)
bench::mark(max(abs($X), abs($Y))) %>%
  print(width = Inf)
""";

# A tibble: 1 × 13
  expression                           min   median `itr/sec` mem_alloc `gc/sec`
  <bch:expr>                      <bch:tm> <bch:tm>     <dbl> <bch:byt>    <dbl>
1 max(abs(`#JL`$X), abs(`#JL`$Y))    1.6ms   1.71ms      565.    15.3MB     279.
  n_itr  n_gc total_time result    memory             time            
  <int> <dbl>   <bch:tm> <list>    <list>             <list>          
1   150    74      265ms <dbl [1]> <Rprofmem [2 × 3]> <bench_tm [224]>
  gc                
  <list>            
1 <tibble [224 × 3]>


In Julia, dot operations are fused so no intermediate arrays are created.

In [84]:
# no intermediate arrays created, only result array created
@benchmark max.(abs.($X), abs.($Y))

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  215.417 μs …   6.280 ms  ┊ GC (min … max):  0.00% … 11.81%
 Time  (median):     244.375 μs               ┊ GC (median):     0.00%
 Time  (mean ± σ):   320.107 μs ± 211.139 μs  ┊ GC (mean ± σ):  19.14% ± 20.09%

  ▇█▇▆▅▅▄▃▃▂▂                                  ▂▂▂▂▂▁▁▁▁▁       ▂
  █████████████▇▇▅▅▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▁▁▁▁▃▅████████████████▇▇ █
  215 μs        Histogram: log(frequency) by time       1.02 ms <

 Memory estimate: 7.64 MiB, allocs estimate: 3.

Pre-allocating result array gets rid of memory allocation at all.

In [85]:
# no memory allocation at all!
Z = zeros(size(X)) # zero matrix of same size as X
@benchmark $Z .= max.(abs.($X), abs.($Y)) # .= (vs =) is important!

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  212.750 μs … 474.708 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     231.750 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   242.556 μs ±  31.349 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▂▅▆▅▄▇██▆▄▄▃▃▃▂▃▃▃▂▂▂▂▁▁▁▁▁▁▁ ▁        ▁ ▁                    ▂
  ████████████████████████████████████████▇██▇▇▇▆▇▇▆▇▅▆▅▆▆▅▅▄▆▃ █
  213 μs        Histogram: log(frequency) by time        375 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

**Exercise:** Annotate the broadcasting by `@turbo` and `@tturbo` (multi-threading) and benchmark again.

### Views

[View](https://docs.julialang.org/en/v1/base/arrays/#Views-(SubArrays-and-other-view-types)-1) avoids creating extra copy of matrix data.

In [86]:
Random.seed!(123)
A = randn(1000, 1000)

# sum entries in a sub-matrix
@benchmark sum($A[1:2:500, 1:2:500])

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  26.791 μs …  1.120 ms  ┊ GC (min … max):  0.00% … 87.34%
 Time  (median):     28.750 μs              ┊ GC (median):     0.00%
 Time  (mean ± σ):   35.716 μs ± 60.173 μs  ┊ GC (mean ± σ):  12.54% ±  7.24%

  ▂▆█▄▂▂▃▂▁▁▁▁                                                ▁
  █████████████▇▇▇▆▅▆▅▅▅▅▄▅▃▄▅▃▃▅▇▇▆▇▆▆▆▆▅▅▆▆▅▅▄▄▄▄▃▅▄▅▄▅▄▅▄▅ █
  26.8 μs      Histogram: log(frequency) by time      78.5 μs <

 Memory estimate: 496.08 KiB, allocs estimate: 3.

In [87]:
# view avoids creating a separate sub-matrix
@benchmark sum(@view $A[1:2:500, 1:2:500])

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  54.958 μs … 119.292 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     55.125 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   56.380 μs ±   3.619 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▃ ▃▁       ▁▂ ▁                                             ▁
  ██▇██▇▇▇▇▇▆▆█████▇▇▇▇▆▆▆▆▆▇▇█▇▇▆▆▆▆▆▆▆▅▆▆▆▆▆▆▅▅▅▅▅▅▅▄▅▅▅▅▄▅▄ █
  55 μs         Histogram: log(frequency) by time        72 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

The [`@views`](https://docs.julialang.org/en/v1/base/arrays/#Base.@views) macro, which can be useful in [some operations](https://discourse.julialang.org/t/why-is-a-manual-in-place-addition-so-much-faster-than-and-on-range-indexed-arrays/3302).

In [88]:
@benchmark @views sum($A[1:2:500, 1:2:500])

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  50.708 μs … 151.750 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     55.125 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   56.519 μs ±   4.121 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

              █▄▂▄▁     ▂▂▁▁         ▁                         ▁
  ▆▁▁▁▁▁▁▁▁▁▁▃██████▇▇█▇████████▇▆▆▇███▇█▇█▇██▇▇██▇▇▇▇▆▆▆▇▆▇▆▆ █
  50.7 μs       Histogram: log(frequency) by time      71.3 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.